In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Optimization Solver: функція Qiskit від Q-CTRL Fire Opal
> **Note:** Qiskit Functions — це експериментальна функція, доступна лише користувачам IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вона перебуває у статусі попереднього релізу та може змінюватись.

## Огляд
За допомогою Fire Opal Optimization Solver ти можеш розв'язувати задачі оптимізації утилітарного масштабу на квантовому залізі без потреби в квантовій експертизі. Просто введи визначення задачі на високому рівні — і Solver зробить усе інше. Весь робочий процес є шумостійким і використовує [Fire Opal Performance Management](/guides/q-ctrl-performance-management) «під капотом». Solver стабільно надає точні рішення для класично складних задач навіть у повному масштабі на найбільших QPU від IBM&reg;.

Solver є гнучким і може розв'язувати комбінаторні задачі оптимізації, визначені як цільові функції або довільні графи. Задачі не потрібно відображати на топологію пристрою. Розв'язуються як необмежені, так і обмежені задачі, за умови що обмеження можуть бути сформульовані у вигляді штрафних доданків. Приклади, наведені в цьому посібнику, демонструють, як розв'язати необмежену та обмежену задачу оптимізації утилітарного масштабу, використовуючи різні типи вхідних даних для Solver. Перший приклад охоплює задачу Max-Cut на 156-вузловому 3-регулярному графі, тоді як другий — задачу Minimum Vertex Cover на 50 вузлах, визначену через цільову функцію.

Щоб отримати доступ до Optimization Solver, [звернись до Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Опис функції
Solver повністю оптимізує та автоматизує весь алгоритм — від придушення помилок на рівні заліза до ефективного відображення задачі та замкненої класичної оптимізації. «За лаштунками» конвеєр Solver зменшує помилки на кожному етапі, забезпечуючи підвищену продуктивність, необхідну для реального масштабування. Базовий робочий процес натхненний алгоритмом квантової наближеної оптимізації (QAOA) — гібридним квантово-класичним алгоритмом. Детальний опис повного робочого процесу Optimization Solver наведено в [опублікованій статті](https://arxiv.org/abs/2406.01743).

![Візуалізація робочого процесу Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Щоб розв'язати загальну задачу за допомогою Optimization Solver:
1. Визнач свою задачу як цільову функцію, граф або `SparsePauliOp` спінового ланцюжка.
2. Підключись до функції через Qiskit Functions Catalog.
3. Запусти задачу на Solver і отримай результати.
## Входи та виходи
### Входи
| Назва    | Тип                    | Опис                                                                                                                                                                                         | Обов'язковий | За замовчуванням | Приклад |
| ---------  |-------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------| -------- | ---------- | ---------- |
| problem  | `str` або `SparsePauliOp` | Одне з представлень, зазначених у розділі «Підтримувані формати задач»                                                                                                                                  | Так | Немає   |`Poly(2.0*n[0]*n[1] + 2.0*n[0]*n[2] - 3.13520241298341*n[0] + 2.0*n[1]*n[2] - 3.14469748506794*n[1] - 3.18897660121566*n[2] + 6.0, n[0], n[1], n[2], domain='RR')` |
| problem_type  | `str`                   | Назва класу задачі; використовується лише для визначень задач на графі та спіновому ланцюжку, які обмежені значеннями "maxcut" або "spin_chain"; не потрібний для довільних цільових функцій | Ні      | `None`| `"maxcut"`      |
| backend_name  | `str`                   | Назва бекенду для використання                                                                                                                                                                          | Ні  | Найменш завантажений бекенд, доступний твоєму інстансу    | `"ibm_fez"`      |
| options  | `dict`                  | Вхідні параметри, зокрема: (необов'язковий) `session_id`: `str`; за замовчуванням створюється нова сесія                                                                                      | Ні | `None`    | `{"session_id": "cw2q70c79ws0008z6g4g"}`     |

**Підтримувані формати задач:**
- Представлення цільової функції у вигляді поліноміального виразу. В ідеалі — створений у Python на основі існуючого об'єкта SymPy Poly та відформатований у рядок за допомогою [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Представлення задачі у вигляді графа для конкретного типу задачі. Граф слід створювати за допомогою бібліотеки networkx у Python, а потім перетворювати на рядок функцією networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Представлення задачі у вигляді спінового ланцюжка. Спіновий ланцюжок має бути представлений як об'єкт `SparsePauliOp`; детальніше — у [документації](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp).

**Підтримувані бекенди:**
Виконай наступний код, щоб переглянути список бекендів, які підтримуються на даний момент. Якщо твій пристрій не вказано у списку, [звернись до Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com), щоб додати підтримку.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>]

**Options:**
| Name   | Type   | Description  | Default |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | An existing Qiskit Runtime session ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | A list of job tags | `[]`|

### Outputs
| Name    | Type | Description | Example |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution and metadata listed under "Result dictionary contents"         | `{'solution_bitstring_cost': 3.0, ‘final_bitstring_distribution’: {‘000001’: 100, ‘000011’: 2}, ‘iteration_count’: 3, 'solution_bitstring': ‘000001’,  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |


**Result dictionary contents:**
| Name    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | The best minimum cost across all iterations        |
| final_bitstring_distribution  | `CountsDict`              | The bitstring counts dictionary associated with the minimum cost        |
| solution_bitstring | `str`              | Bitstring with the best cost in the final distribution         |
| iteration_count  | `int`              | The total number of QAOA iterations performed by the Solver        |
| variables_to_bitstring_index_map  | `float`              | The mapping from the variables to the equivalent bit in the bitstring       |
| best_parameters  | `list[float]`            | The optimized beta and gamma parameters across all iterations  |
| warnings  |`list[str]`              | The warnings produced while compiling or running QAOA (defaults to None)   |

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected Max-Cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

**Параметри:**
| Назва   | Тип   | Опис  | За замовчуванням |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | Ідентифікатор існуючої сесії Qiskit Runtime  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | Список тегів завдань | `[]`|

### Виходи
| Назва    | Тип | Опис | Приклад |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Рішення та метадані, перелічені у розділі «Вміст словника результатів»         | `{'solution_bitstring_cost': 3.0, 'final_bitstring_distribution': {'000001': 100, '000011': 2}, 'iteration_count': 3, 'solution_bitstring': '000001',  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |

**Вміст словника результатів:**
| Назва    | Тип | Опис |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | Найкраща мінімальна вартість по всіх ітераціях        |
| final_bitstring_distribution  | `CountsDict`              | Словник лічильників бітових рядків, що відповідає мінімальній вартості        |
| solution_bitstring | `str`              | Бітовий рядок із найкращою вартістю в фінальному розподілі         |
| iteration_count  | `int`              | Загальна кількість ітерацій QAOA, виконаних Solver        |
| variables_to_bitstring_index_map  | `float`              | Відображення змінних на відповідні біти у бітовому рядку       |
| best_parameters  | `list[float]`            | Оптимізовані параметри beta та gamma по всіх ітераціях  |
| warnings  |`list[str]`              | Попередження, що виникли під час компіляції або виконання QAOA (за замовчуванням None)   |
## Бенчмарки
[Опубліковані результати бенчмаркінгу](https://arxiv.org/abs/2406.01743) показують, що Solver успішно розв'язує задачі на понад 120 кубітах, навіть перевершуючи раніше опубліковані результати на пристроях квантового відпалу та пастки іонів. Наступні метрики бенчмарку дають приблизне уявлення про точність і масштабування типів задач на основі кількох прикладів. Фактичні метрики можуть відрізнятись залежно від різних характеристик задачі, таких як кількість доданків у цільовій функції (щільність) та їхня локальність, кількість змінних і порядок полінома.

«Кількість кубітів», що вказана, не є жорстким обмеженням, а лише приблизними порогами, при яких можна очікувати надзвичайно стабільної точності рішення. Задачі більшого розміру вже були успішно розв'язані, і тестування за межами цих порогів заохочується.

Довільне з'єднання кубітів підтримується для всіх типів задач.

| Тип задачі    | Кількість кубітів | Приклад | Точність | Загальний час (с) | Використання Runtime (с) | Кількість ітерацій
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Розріджено-зв'язані квадратичні задачі  | 156 | 3-регулярний Max-Cut| 100%     | 1764     | 293          | 16 |
| Бінарна оптимізація вищого порядку | 156 | Ізингова модель спінового скла | 100%      | 1461     | 272          | 16 |
| Щільно-зв'язані квадратичні задачі | 50 | Повністю-зв'язаний Max-Cut| 100%      |  1758    | 268  | 12 |
| Обмежена задача зі штрафними доданками | 50 | Зважений Minimum Vertex Cover із щільністю ребер 8% | 100%      | 1074     | 215 | 10 |
## Початок роботи
Спочатку пройди автентифікацію за допомогою свого [API-ключа IBM Quantum](http://quantum.cloud.ibm.com/). Потім вибери функцію Qiskit наступним чином. (Цей фрагмент передбачає, що ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі.)

In [2]:
# %pip install networkx numpy

## Приклад: необмежена оптимізація
Запусти задачу [максимального розрізу](https://en.wikipedia.org/wiki/Maximum_cut) (Max-Cut). Наступний приклад демонструє можливості Solver на задачі Max-Cut на 156-вузловому, 3-регулярному незваженому графі, але ти також можеш розв'язувати задачі на зважених графах.
Окрім `qiskit-ibm-catalog`, для виконання цього прикладу тобі знадобляться такі пакети: `networkx` і `numpy`. Ти можеш встановити їх, розкоментувавши наступну комірку, якщо запускаєш цей приклад у ноутбуці з ядром IPython.

In [2]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [3]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

The Solver accepts a string as the problem definition input.

In [4]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

![Вихід попередньої комірки коду](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif)

Solver приймає рядок як вхідне визначення задачі.

In [9]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [14]:
# Get job status
print(maxcut_job.status())

QUEUED


Перевір [статус](/guides/functions#check-job-status) робочого навантаження своєї функції Qiskit або отримай [результати](/guides/functions#retrieve-results) наступним чином:

In [15]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 209.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior Max-Cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [ ]:
# %pip install numpy networkx sympy

### 3. Отримайте результат
Отримай оптимальне значення розрізу зі словника результатів.

> **Note:** Відображення змінних на бітовий рядок може змінитись. Вихідний словник містить підсловник `variables_to_bitstring_index_map`, який допомагає перевірити порядок.

In [ ]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

A standard optimization model for weighted MVC can be formulated as follows. First, a penalty must be added for any case where an edge is not connected to a vertex in the subset. Therefore, let $n_i = 1$ if vertex $i$ is in the cover (i.e., in the subset) and $n_i = 0$ otherwise. Second, the goal is to minimize the total number of vertices in the subset, which can be represented by the following function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Ти можеш перевірити точність результату, розв'язавши задачу класично за допомогою відкритих солверів, таких як [PuLP](https://coin-or.github.io/pulp/), якщо граф не є щільно зв'язаним. Для задач із високою щільністю може знадобитись використання просунутих класичних солверів для підтвердження рішення.
## Приклад: обмежена оптимізація
Попередній приклад Max-Cut є класичною квадратичною необмеженою задачею бінарної оптимізації. Optimization Solver від Q-CTRL може використовуватись для різних типів задач, зокрема обмеженої оптимізації. Ти можеш розв'язувати довільні типи задач, подаючи визначення задачі у вигляді полінома, де обмеження моделюються як штрафні доданки.

Наступний приклад демонструє, як побудувати цільову функцію для обмеженої задачі оптимізації — [мінімального покриття вершин](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Окрім пакетів `qiskit-ibm-catalog` і `qiskit`, для виконання цього прикладу тобі знадобляться такі пакети: `numpy`, `networkx` і `sympy`. Ти можеш встановити їх, розкоментувавши наступну комірку, якщо запускаєш цей приклад у ноутбуці з ядром IPython.

In [ ]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

### 1. Визначте задачу
Визнач випадкову задачу MVC, згенерувавши граф із вузлами, що мають випадкові ваги.

In [ ]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

![Вихід попередньої комірки коду](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif)

Стандартна модель оптимізації для зваженого MVC може бути сформульована наступним чином. Спочатку необхідно додати штраф для будь-якого випадку, коли ребро не з'єднане з вершиною з підмножини. Тому нехай $n_i = 1$, якщо вершина $i$ входить до покриття (тобто знаходиться у підмножині), і $n_i = 0$ в іншому випадку. По-друге, мета полягає у мінімізації загальної кількості вершин у підмножині, що може бути виражено такою функцією:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
print(mvc_job.status())

Тепер кожне ребро в графі повинне мати хоча б одну кінцеву точку з покриття, що можна виразити як нерівність:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Будь-який випадок, коли ребро не з'єднане з вершиною покриття, має бути оштрафований. Це можна представити у цільовій функції, додавши штрафний доданок виду $P(1-n_i-n_j+n_i n_j)$, де $P$ — це позитивна штрафна константа. Таким чином, необмежена альтернатива обмеженій нерівності для зваженого MVC виглядає так:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [ ]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


### 2. Запустіть задачу